In [ ]:
from bmadx.distgen_utils import create_beam
from bmadx.plot import plot_projections
from bmadx.constants import PI
import os
import torch
from bmadx.bmad_torch.track_torch import particle_to_beam


n_part = 100_000
p0c = 245596401.66703767
import numpy as np



In [ ]:
from bmadx import Particle, M_ELECTRON
#par_in = os.path.join("data", "partcl_GPSR_BC3H")

f2 = open('./partcl_GPSR_BC1_Linear1.data', 'r')


ff2 = f2.readlines()
f2.close()

# Empty list
x = []
y = []
px= []
py= []
z = []
pz = [] 
for fi in ff2:
    token = fi.split()
    try:
        float(token[0])
        x.append(float(token[0]))
        px.append(float(token[1]))
        y.append(float(token[2]))
        py.append(float(token[3]))
        z.append(float(token[4]))
        pz.append(float(token[5]))
    except:
        continue
x = np.array(x)
y = np.array(y)
z = np.array(z)
px= np.array(px)
py= np.array(py)
pz= np.array(pz)

x = x - np.mean(x)
y = y - np.mean(y)
z = z - np.mean(z)
px = px - np.mean(px)
py = py - np.mean(py)

coords = np.array([x, px, y, py, z, pz])
s = 0

par_in = Particle(*coords, s, p0c, M_ELECTRON)
par_in

In [ ]:
gt_beam = particle_to_beam(par_in)

In [ ]:
emitx = np.sqrt( np.mean((x-np.mean(x))**2)*np.mean((px-np.mean(px))**2) - np.mean((x-np.mean(x))*(px-np.mean(px)))**2)
emity = np.sqrt( np.mean((y-np.mean(y))**2)*np.mean((py-np.mean(py))**2) - np.mean((y-np.mean(y))*(py-np.mean(py)))**2)
emitz = np.sqrt( np.mean((z-np.mean(z))**2)*np.mean((pz-np.mean(pz))**2) - np.mean((z-np.mean(z))*(pz-np.mean(pz)))**2)

# Normalized emittance
enx = emitx * (p0c/0.511e6)
eny = emity * (p0c/0.511e6)
enz = emitz * (p0c/0.511e6)

sigx = np.sqrt( np.mean( ((x - np.mean(x))**2 )))
sigy = np.sqrt( np.mean( ((y - np.mean(y))**2 )))
sigz = np.sqrt( np.mean( ((z - np.mean(z))**2 )))

# =====================================================================
# =====================================================================
# Linear Twiss parameters from this setting
betax = (sigx**2) / emitx # since sigx is in mm unit
betay = (sigy**2) / emity

alphax= - (np.mean((x-np.mean(x))*(px-np.mean(px)))) / emitx
alphay= - (np.mean((y-np.mean(y))*(py-np.mean(py)))) / emity

print(enx)
print(eny)
print(enz)
print(sigx)
print(sigy)
print(sigz)

In [ ]:
par_in.z.shape

In [ ]:
#plot gt beam
lims = np.array([[-1e-3, 1e-3],
                 [-0.1e-3, 0.1e-3],
                 [-0.5e-3, 0.5e-3],
                 [-0.05e-3, 0.05e-3],
                 [-2e-3, 2e-3],
                 [-1e-2, 1e-2]])*1
fig, ax = plot_projections(
    gt_beam.numpy_particles(),
    custom_lims = lims,
    background = 0
)

In [ ]:
# ===========================================================================================================================================
# ===========================================================================================================================================
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from phase_space_reconstruction.virtual.beamlines import palxfel_Simulation
from phase_space_reconstruction.diagnostics import ImageDiagnostic
from phase_space_reconstruction.train import train_3d_scan_palxfel_Simulation

from bmadx.plot import plot_projections
from bmadx.constants import PI
from bmadx.bmad_torch.track_torch import particle_to_beam



In [ ]:
# diagnostic beamline:
lattice_scm1, lattice_scm2 = palxfel_Simulation(p0c=p0c, dipole_on=True)

In [ ]:
lattice_scm1

In [ ]:
lattice_scm2

In [ ]:
from bmadx import PI
# Scan over quad strength, tdc on/off and dipole on/off
scan_ids = [0, 2, 4, 6, 8, 10, 14] 


n_ks = 7
ks = torch.linspace(-5, 5, n_ks) # quad ks
vs = torch.tensor([-79.2477, -76.1061, -72.9645, -69.8229, -66.6814, -63.5398, -60.3982]) * PI/180 / (2*PI)
gs = torch.ones(len(vs))*232861950.55756843 # Reference beam energy at the dipole magnets
ts = torch.ones(len(vs))*13.5E6 # cavity gradient
ks2= torch.ones(len(vs))*3.5 # quad coefficient
ks3= torch.ones(len(vs))*1E-16 # quad coefficient
ks4= torch.ones(len(vs))*1E-16 # quad coefficient

vs = torch.cat((torch.ones(1)*0, vs))
gs = torch.cat((torch.ones(1)*p0c, gs))
ts = torch.cat((torch.ones(1)*3E-16, ts))
ks2 = torch.cat((torch.ones(1)*ks2[0], ks2))
ks3 = torch.cat((torch.ones(1)*ks3[0], ks3))
ks4 = torch.cat((torch.ones(1)*ks4[0], ks4))


train_params  = torch.meshgrid(ks, vs, indexing="ij")
params  = torch.stack(train_params, dim=-1).reshape((-1, 2))
print(len(params))



# Re-define the parameter for exact cases
params = torch.cat(( params.squeeze(), 
                    torch.Tensor.repeat(gs,len(ks)).unsqueeze(-1), 
                    torch.Tensor.repeat(ts,len(ks)).unsqueeze(-1),
                    torch.Tensor.repeat(ks2,len(ks)).unsqueeze(-1),
                    torch.Tensor.repeat(ks3,len(ks)).unsqueeze(-1),
                    torch.Tensor.repeat(ks4,len(ks)).unsqueeze(-1)),1).unsqueeze(-1)

In [ ]:

# create diagnostic screen: 
bins2 = torch.linspace(-5, 5, 500) * 1e-3
bandwidth2 = (bins2[1]-bins2[0]) / 2
screen2 = ImageDiagnostic(bins2, bins2, bandwidth2)
resolution2 = bandwidth2*2
print(resolution2)


In [ ]:
# generate and save train and test datasets
save_dir = os.path.join('./')
from phase_space_reconstruction.virtual.scans import run_palxfel_Simulation

gt_dset = run_palxfel_Simulation(
    gt_beam,
    lattice_scm1,
    lattice_scm2,
    screen2,
    params,
    ids = scan_ids,
    save_as = os.path.join(save_dir, 'GroundTruth_Dataset_PALXFEL.dset')
    )




In [ ]:
# load data
dset = gt_dset
print(len(dset.images1))
print(len(dset.images2))

In [ ]:

s = torch.arange(0,20,1).to(device='cuda')
fig,ax = plt.subplots(len(s),2,sharex="all", sharey="all", gridspec_kw={"hspace":0.01,"wspace":0.05,"right":0.97,"top":2.5,"bottom":1})
fig.set_size_inches(10,30)
for i in range(len(s)):
    ax[i][0].imshow((np.transpose(dset.images1[2*i][0])))
    ax[i][1].imshow((np.transpose(dset.images2[2*i][0])))


In [ ]:
%%time
save_dir = './'

from phase_space_reconstruction.train import train_3d_scan_palxfel_Simulation

pred_beam_3d_scan_10_000, model = train_3d_scan_palxfel_Simulation(
    dset, 
    lattice_scm1,
    lattice_scm2,
    p0c, 
    screen2, 
    screen2, 
    ids = scan_ids,
    n_epochs = 2_000, 
    n_particles = 100_000, 
    device = 'cuda',
    save_dir = save_dir,
    distribution_dump_frequency=200,
    distribution_dump_n_particles=500_000,
    use_decay=False,
    lr=0.01,
    )

In [ ]:
import torch
from stats import plot_projections_with_contours, show_cov_stats
import os


lims = np.array([[-1e-3, 1e-3],
                 [-0.1e-3, 0.1e-3],
                 [-0.5e-3, 0.5e-3],
                 [-0.05e-3, 0.05e-3],
                 [-2e-3, 2e-3],
                 [-1e-3, 1e-3]])*1E3

recn_dist = torch.load('3d_scan_result.pt', weights_only=False)
recn_dist.data = recn_dist.data
fig,ax = plot_projections_with_contours(
    recn_dist,
    gt_beam,
    n_bins=200,
    contour_percentiles = [50, 90],
    custom_lims=lims,
    contour_smoothing=0.25,
)

In [ ]:
X  = recn_dist.x
xp = recn_dist.px
Y  = recn_dist.y
yp = recn_dist.py
Z  = recn_dist.z
delta = recn_dist.pz



In [ ]:
torch.set_printoptions(precision=9)

emitx = torch.sqrt( torch.mean((X-torch.mean(X))**2)*torch.mean((xp-torch.mean(xp))**2) - torch.mean((X-torch.mean(X))*(xp-torch.mean(xp)))**2)
emity = torch.sqrt( torch.mean((Y-torch.mean(Y))**2)*torch.mean((yp-torch.mean(yp))**2) - torch.mean((Y-torch.mean(Y))*(yp-torch.mean(yp)))**2)
emitz = torch.sqrt( torch.mean((Z-torch.mean(Z))**2)*torch.mean((delta-torch.mean(delta))**2) - torch.mean((Z-torch.mean(Z))*(delta-torch.mean(delta)))**2)

# Normalized emittance
enx = emitx * (p0c/0.511e6)
eny = emity * (p0c/0.511e6)
enz = emitz * (p0c/0.511e6)

sigx = torch.sqrt( torch.mean( ((X - torch.mean(X))**2 )))
sigy = torch.sqrt( torch.mean( ((Y - torch.mean(Y))**2 )))
sigz = torch.sqrt( torch.mean( ((Z - torch.mean(Z))**2 )))

# =====================================================================
# =====================================================================
# Linear Twiss parameters from this setting
betax = (sigx**2) / emitx # since sigx is in mm unit
betay = (sigy**2) / emity

alphax= - (torch.mean((X-torch.mean(X))*(xp-torch.mean(xp)))) / emitx
alphay= - (torch.mean((Y-torch.mean(Y))*(yp-torch.mean(yp)))) / emity

print(enx)
print(eny)
print(enz)
print(sigx)
print(sigy)
print(sigz)

In [ ]:
# generate and save train and test datasets
save_dir = os.path.join('./')
from phase_space_reconstruction.virtual.scans import run_palxfel_Simulation

recon_dset = run_palxfel_Simulation(
    recn_dist,
    lattice_scm1,
    lattice_scm2,
    screen2,
    params,
    ids = scan_ids,
    save_as = os.path.join(save_dir, 'prediction_dataset.dset')
    )




In [ ]:

s = torch.arange(0,11,1).to(device='cuda')
fig,ax = plt.subplots(len(s),2,sharex="all", sharey="all", gridspec_kw={"hspace":0.02,"wspace":0.05,"right":0.97,"top":2.5,"bottom":1})
fig.set_size_inches(10,30)
for i in range(len(s)):
    ax[i][0].imshow((np.transpose(dset.images1[2*i][0])))
    ax[i][1].imshow((np.transpose(recon_dset.images1[2*i][0])))


In [ ]:

s = torch.arange(0,11,1).to(device='cuda')
fig,ax = plt.subplots(len(s),2,sharex="all", sharey="all", gridspec_kw={"hspace":0.02,"wspace":0.05,"right":0.97,"top":2.5,"bottom":1})
fig.set_size_inches(10,30)
for i in range(len(s)):
    ax[i][0].imshow((np.transpose(dset.images2[3*i][0])))
    ax[i][1].imshow((np.transpose(recon_dset.images2[3*i][0])))
